# H5N1 Tensor SVD Analysis & Embedding Visualisation
This notebook loads the SVD decomposition results from the FAMSA tensor pipeline, plots scree plots, and visualises segment/sample loadings aggregated by continent and year.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

svd_dir = Path("/data/users/ltucker/influenzaData/H5N1_pipeline/output/famsa_tensor_analysis/svd")

segment_names = ["PB2", "PB1", "PA", "HA", "NP", "NA", "MP", "NS"]
mode_names = ["Mode 0: Segments", "Mode 1: Samples (Rows)", "Mode 2: Samples (Cols)"]
n_components_show = 5  # how many PCs to inspect in loadings plots

## 1. Load SVD Results

In [ ]:
svd = {}
variance_dfs = {}

for mode in range(3):
    svd[mode] = {
        "U":  np.load(svd_dir / f"U_mode{mode}.npy"),
        "S":  np.load(svd_dir / f"S_mode{mode}.npy"),
        "Vt": np.load(svd_dir / f"Vt_mode{mode}.npy"),
    }
    variance_dfs[mode] = pd.read_parquet(svd_dir / f"variance_mode{mode}.parquet")

    n_sv = len(svd[mode]["S"])
    df = variance_dfs[mode]
    n90 = df.loc[df["cumulative_variance"] >= 0.90, "component"].iloc[0] if (df["cumulative_variance"] >= 0.90).any() else "N/A"
    n95 = df.loc[df["cumulative_variance"] >= 0.95, "component"].iloc[0] if (df["cumulative_variance"] >= 0.95).any() else "N/A"
    print(f"{mode_names[mode]}: {n_sv} singular values | 90% at PC{n90} | 95% at PC{n95}")

# Load sample IDs
sample_ids = pd.read_parquet(svd_dir / "sample_ids.parquet")["sample_id"].tolist()
print(f"\nTotal samples: {len(sample_ids)}")

## 2. Parse Metadata

In [ ]:
# Format: A/H5N0|A/chicken/Fujian/...|PB2|1|EPI_ISL_...|china|asia|2018
metadata = pd.DataFrame({"sample_id": sample_ids})
split = metadata["sample_id"].str.split("|")
metadata["country"]   = split.str[5]
metadata["continent"] = split.str[6]
metadata["year"]      = pd.to_numeric(split.str[7], errors="coerce")

print(f"Samples: {len(metadata)}")
print(f"Countries: {metadata['country'].nunique()} | Continents: {metadata['continent'].nunique()}")
print(f"Years: {int(metadata['year'].min())}\u2013{int(metadata['year'].max())}")

## 3. Scree Plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for mode in range(3):
    df = variance_dfs[mode]
    max_show = len(df) if mode == 0 else min(50, len(df))
    d = df.iloc[:max_show]
    suffix = f" (first {max_show})" if mode > 0 else ""

    axes[0, mode].plot(d["component"], d["singular_value"], "o-", markersize=4)
    axes[0, mode].set_xlabel("Component")
    axes[0, mode].set_ylabel("Singular Value")
    axes[0, mode].set_title(f"{mode_names[mode]} \u2014 Singular Values{suffix}")
    axes[0, mode].grid(True, alpha=0.3)

    axes[1, mode].plot(d["component"], d["cumulative_variance"], "o-", markersize=4)
    axes[1, mode].axhline(y=0.90, color="r", linestyle="--", label="90%")
    axes[1, mode].axhline(y=0.95, color="orange", linestyle="--", label="95%")
    axes[1, mode].set_xlabel("Number of Components")
    axes[1, mode].set_ylabel("Cumulative Variance Explained")
    axes[1, mode].set_title(f"{mode_names[mode]} \u2014 Cumulative Variance{suffix}")
    axes[1, mode].legend()
    axes[1, mode].grid(True, alpha=0.3)

plt.suptitle("Scree Plots \u2014 SVD on Tensor Unfoldings", fontsize=16, y=1.00)
plt.tight_layout()
plt.show()

## 4. Mode 0 2014 Segment Loadings

In [ ]:
U0 = svd[0]["U"]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for pc in range(min(8, U0.shape[1])):
    ax = axes[pc]
    loadings = U0[:, pc]
    colors = ["salmon" if v > 0 else "skyblue" for v in loadings]
    ax.bar(segment_names, loadings, color=colors, alpha=0.9)
    ax.axhline(y=0, color="black", linewidth=0.5)
    ax.set_ylabel("Loading")
    ax.set_title(f"PC{pc+1}")
    ax.grid(True, alpha=0.3, axis="y")
    ax.tick_params(axis="x", rotation=45)

plt.suptitle("Mode 0: Segment Loadings", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Mode 1 \u2014 Sample Loadings by Continent

In [ ]:
U1 = svd[1]["U"]
pc_cols = [f"PC{i+1}" for i in range(n_components_show)]
sample_loadings = pd.DataFrame(U1[:, :n_components_show], columns=pc_cols)
loadings_df = pd.concat([metadata.reset_index(drop=True), sample_loadings], axis=1)

grouped_continent = loadings_df.groupby("continent")[pc_cols].sum()

fig, axes = plt.subplots(1, n_components_show + 1, figsize=(4 * (n_components_show + 1), 5))

for i in range(n_components_show):
    col = pc_cols[i]
    vals = grouped_continent[col]
    colors = ["salmon" if v > 0 else "skyblue" for v in vals]
    axes[i].bar(vals.index, vals.values, color=colors, alpha=0.9)
    axes[i].axhline(y=0, color="black", linewidth=0.5)
    axes[i].set_title(col)
    axes[i].set_ylabel("Sum of loadings")
    axes[i].tick_params(axis="x", rotation=45)
    axes[i].grid(True, alpha=0.3, axis="y")

counts = metadata.groupby("continent").size()
axes[-1].bar(counts.index, counts.values, color="steelblue", alpha=0.8)
axes[-1].set_title("Sample Count")
axes[-1].tick_params(axis="x", rotation=45)
axes[-1].grid(True, alpha=0.3, axis="y")

plt.suptitle("Mode 1: Sample Loadings by Continent", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Mode 1 \u2014 Sample Loadings by Year

In [ ]:
grouped_year = loadings_df.groupby("year")[pc_cols].sum()
year_counts = metadata.groupby("year").size()

fig, axes = plt.subplots(2, n_components_show + 1, figsize=(4 * (n_components_show + 1), 8))

for i in range(n_components_show):
    col = pc_cols[i]
    # Top row: raw
    axes[0, i].scatter(grouped_year.index, grouped_year[col], s=15, alpha=0.8)
    axes[0, i].set_title(col)
    axes[0, i].set_ylabel("Sum of loadings")
    axes[0, i].grid(True, alpha=0.3)
    axes[0, i].tick_params(axis="x", rotation=45)
    # Bottom row: symlog scale
    axes[1, i].scatter(grouped_year.index, grouped_year[col], s=15, alpha=0.8)
    axes[1, i].set_yscale("symlog")
    axes[1, i].set_title(f"{col} (symlog)")
    axes[1, i].set_ylabel("Sum of loadings")
    axes[1, i].grid(True, alpha=0.3)
    axes[1, i].tick_params(axis="x", rotation=45)

axes[0, -1].scatter(year_counts.index, year_counts.values, s=15, color="steelblue")
axes[0, -1].set_title("Sample Count")
axes[0, -1].tick_params(axis="x", rotation=45)
axes[0, -1].grid(True, alpha=0.3)

axes[1, -1].scatter(year_counts.index, np.log1p(year_counts.values), s=15, color="steelblue")
axes[1, -1].set_title("Log Sample Count")
axes[1, -1].tick_params(axis="x", rotation=45)
axes[1, -1].grid(True, alpha=0.3)

plt.suptitle("Mode 1: Sample Loadings by Year", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Mode 2 \u2014 Sample Loadings by Continent

In [ ]:
U2 = svd[2]["U"]
sample_loadings_m2 = pd.DataFrame(U2[:, :n_components_show], columns=pc_cols)
loadings_m2 = pd.concat([metadata.reset_index(drop=True), sample_loadings_m2], axis=1)

grouped_continent_m2 = loadings_m2.groupby("continent")[pc_cols].sum()

fig, axes = plt.subplots(1, n_components_show + 1, figsize=(4 * (n_components_show + 1), 5))

for i in range(n_components_show):
    col = pc_cols[i]
    vals = grouped_continent_m2[col]
    colors = ["salmon" if v > 0 else "skyblue" for v in vals]
    axes[i].bar(vals.index, vals.values, color=colors, alpha=0.9)
    axes[i].axhline(y=0, color="black", linewidth=0.5)
    axes[i].set_title(col)
    axes[i].set_ylabel("Sum of loadings")
    axes[i].tick_params(axis="x", rotation=45)
    axes[i].grid(True, alpha=0.3, axis="y")

counts = metadata.groupby("continent").size()
axes[-1].bar(counts.index, counts.values, color="steelblue", alpha=0.8)
axes[-1].set_title("Sample Count")
axes[-1].tick_params(axis="x", rotation=45)
axes[-1].grid(True, alpha=0.3, axis="y")

plt.suptitle("Mode 2: Sample Loadings by Continent", fontsize=14)
plt.tight_layout()
plt.show()

## 8. Mode 2 2014 Sample Loadings by Year

In [ ]:
grouped_year_m2 = loadings_m2.groupby("year")[pc_cols].sum()

fig, axes = plt.subplots(2, n_components_show + 1, figsize=(4 * (n_components_show + 1), 8))

for i in range(n_components_show):
    col = pc_cols[i]
    axes[0, i].scatter(grouped_year_m2.index, grouped_year_m2[col], s=15, alpha=0.8)
    axes[0, i].set_title(col)
    axes[0, i].set_ylabel("Sum of loadings")
    axes[0, i].grid(True, alpha=0.3)
    axes[0, i].tick_params(axis="x", rotation=45)

    axes[1, i].scatter(grouped_year_m2.index, grouped_year_m2[col], s=15, alpha=0.8)
    axes[1, i].set_yscale("symlog")
    axes[1, i].set_title(f"{col} (symlog)")
    axes[1, i].set_ylabel("Sum of loadings")
    axes[1, i].grid(True, alpha=0.3)
    axes[1, i].tick_params(axis="x", rotation=45)

year_counts = metadata.groupby("year").size()
axes[0, -1].scatter(year_counts.index, year_counts.values, s=15, color="steelblue")
axes[0, -1].set_title("Sample Count")
axes[0, -1].tick_params(axis="x", rotation=45)
axes[0, -1].grid(True, alpha=0.3)

axes[1, -1].scatter(year_counts.index, np.log1p(year_counts.values), s=15, color="steelblue")
axes[1, -1].set_title("Log Sample Count")
axes[1, -1].tick_params(axis="x", rotation=45)
axes[1, -1].grid(True, alpha=0.3)

plt.suptitle("Mode 2: Sample Loadings by Year", fontsize=14)
plt.tight_layout()
plt.show()